# 04 - Federated Learning Setup

## Overview

Setup Flower framework, create non-IID client partitions using Dirichlet distribution, and prepare XGBoost/LightGBM NumPyClients.

**Output**: Client partitions, Flower server/client code

In [ ]:
# Imports
import numpy as np
import pandas as pd
import os
import json
import warnings
warnings.filterwarnings('ignore')

import sys
sys.path.insert(0, '/home/willtanoe/Documents/fl-xgb-thesis')
from src.config import (
    NUM_CLIENTS, DIRICHLET_ALPHA, SEED,
    XGB_PARAMS, LGB_PARAMS
)

# Paths
BASE = "/home/willtanoe/Documents/fl-xgb-thesis"
PREPROCESSED_DIR = f"{BASE}/results/preprocessed"
CLIENTS_DIR = f"{BASE}/results/clients"
FIGURES_DIR = f"{BASE}/results/figures"

os.makedirs(CLIENTS_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

print(f"Num clients: {NUM_CLIENTS}")
print(f"Dirichlet alpha: {DIRICHLET_ALPHA}")
print(f"Clients dir: {CLIENTS_DIR}")

## 1. Load Preprocessed Data

In [ ]:
print("Loading preprocessed data...")
train_df = pd.read_parquet(os.path.join(PREPROCESSED_DIR, "train.parquet"))

# Load metadata
with open(os.path.join(PREPROCESSED_DIR, "metadata.json"), 'r') as f:
    metadata = json.load(f)

feature_cols = metadata['feature_cols']
num_classes = metadata['num_classes']

print(f"Train shape: {train_df.shape}")
print(f"Features: {len(feature_cols)}")
print(f"Classes: {num_classes}")

## 2. Create Non-IID Client Partitions (Dirichlet)

In [ ]:
def create_dirichlet_partitions(X, y, num_clients, alpha, seed=42):
    """
    Create non-IID client partitions using Dirichlet distribution.
    
    Args:
        X: Feature array
        y: Label array
        num_clients: Number of clients
        alpha: Dirichlet concentration parameter (lower = more non-IID)
        seed: Random seed
    
    Returns:
        dict: client_id -> (X_client, y_client)
    """
    np.random.seed(seed)
    
    classes = np.unique(y)
    num_classes = len(classes)
    
    # Generate Dirichlet distribution for each class
    # Dirichlet(alpha) gives proportions that sum to 1
    class_proportions = np.random.dirichlet(
        [alpha] * num_clients,  # concentration for each client
        num_classes             # one distribution per class
    )
    
    # Assign samples to clients based on class proportions
    client_data = {i: {'X': [], 'y': []} for i in range(num_clients)}
    
    for cls_idx, cls in enumerate(classes):
        cls_indices = np.where(y == cls)[0]
        np.random.shuffle(cls_indices)
        
        # Split this class's samples across clients
        splits = np.split(cls_indices, 
                         np.cumsum(class_proportions[cls_idx] * len(cls_indices))[:-1].astype(int))
        
        for client_id, split in enumerate(splits):
            client_data[client_id]['X'].append(X[split])
            client_data[client_id]['y'].append(y[split])
    
    # Concatenate for each client
    result = {}
    for client_id in range(num_clients):
        X_client = np.vstack(client_data[client_id]['X'])
        y_client = np.concatenate(client_data[client_id]['y'])
        result[client_id] = (X_client, y_client)
    
    return result

print("Creating Dirichlet partitions...")
X_train = train_df[feature_cols].values
y_train = train_df['label'].values

client_partitions = create_dirichlet_partitions(
    X_train, y_train, 
    num_clients=NUM_CLIENTS,
    alpha=DIRICHLET_ALPHA,
    seed=SEED
)

# Show partition sizes
print("\nClient partition sizes:")
for cid, (X_c, y_c) in client_partitions.items():
    print(f"  Client {cid}: {X_c.shape[0]:,} samples, {len(np.unique(y_c))} classes")

## 3. Visualize Non-IID Distribution

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Create heatmap of class distribution per client
distribution = np.zeros((NUM_CLIENTS, num_classes))
for cid, (X_c, y_c) in client_partitions.items():
    unique, counts = np.unique(y_c, return_counts=True)
    for cls, cnt in zip(unique, counts):
        distribution[cid, cls] = cnt

# Normalize per client
distribution_norm = distribution / distribution.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Absolute counts (log scale)
ax1 = axes[0]
sns.heatmap(np.log1p(distribution), ax=ax1, cmap='YlOrRd', 
            xticklabels=5, yticklabels=5)
ax1.set_title('Client-Class Distribution (Log Scale)')
ax1.set_xlabel('Class ID')
ax1.set_ylabel('Client ID')

# Normalized (proportions)
ax2 = axes[1]
sns.heatmap(distribution_norm, ax=ax2, cmap='YlOrRd',
            xticklabels=5, yticklabels=5, vmin=0, vmax=0.5)
ax2.set_title('Client-Class Proportions (Normalized)')
ax2.set_xlabel('Class ID')
ax2.set_ylabel('Client ID')

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, '04_non_iid_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f"Figure saved: {FIGURES_DIR}/04_non_iid_distribution.png")

## 4. Save Client Partitions

In [ ]:
print("\nSaving client partitions...")

for cid, (X_c, y_c) in client_partitions.items():
    client_dir = os.path.join(CLIENTS_DIR, f"client_{cid}")
    os.makedirs(client_dir, exist_ok=True)
    
    # Save as parquet
    df = pd.DataFrame(X_c, columns=feature_cols)
    df['label'] = y_c
    df.to_parquet(os.path.join(client_dir, "data.parquet"), index=False)
    
    # Save metadata
    client_meta = {
        'client_id': cid,
        'num_samples': int(len(y_c)),
        'num_classes': len(np.unique(y_c)),
        'classes': sorted(np.unique(y_c).tolist())
    }
    with open(os.path.join(client_dir, "metadata.json"), 'w') as f:
        json.dump(client_meta, f, indent=2)
    
    print(f"  Client {cid}: {X_c.shape[0]:,} samples saved")

print(f"\nAll client partitions saved to: {CLIENTS_DIR}")

## 5. Create Flower Client Classes

## 5. Flower Client Classes

The actual Flower client classes (`XGBNumPyClient`, `LGBNumPyClient`) and the `client_fn` factory are defined in:

**`flwr_app/client_app.py`**

These use incremental training via `xgb.train(xgb_model=prev_model)` / `lgb.train(init_model=prev_model)` with Round-Robin strategy. See the Flower app source for details.

## 7. Summary

In [ ]:
print("FL setup complete!")
print(f"\nSummary:")
print(f"  Clients: {NUM_CLIENTS}")
print(f"  Non-IID alpha: {DIRICHLET_ALPHA}")
print(f"  Clients dir: {CLIENTS_DIR}")
print(f"  Flower app client code: flwr_app/client_app.py")